In [1]:
"""Main file for running annealed Langevin dynamics for new material sampling."""
from __future__ import annotations
from pathlib import Path
from types import SimpleNamespace

import torch

from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model_egnn import CHGGen
from chggen.common.data_utils import get_scaler

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = CHGNetDataset(
    path='/home/xzdai/ceder_group/material_dircovery/chggen_old/data/perov_5/test_zpc.csv',
    name = 'A_good_name',
    prop_list = ['heat_all'],
)

100%|██████████| 50/50 [00:00<00:00, 253.04it/s]


In [3]:
lattice_scaler = get_scaler(dataset= dataset)

model_hparams ={'latent_dim': 64, 'hidden_dim': 128, 
                'predict_property': True, 'property_dim': 1,  
                'load_pretrain': True, 
                'fc_num_layers': 1, 
                'sigma_F_begin': 10.0, 'sigma_F_end': 0.01, 
                'sigma_L_begin': 1.0, 'sigma_L_end': 0.01, 
                'type_sigma_begin': 5.0, 'type_sigma_end': 0.01,
                'max_atoms': 20, 
                'num_noise_level': 1, 
                'lattice_scale_method': 'scale_length', 
                'cost_natom': 1.0, 'cost_coord': 10.0, 'cost_type': 1.0, 'cost_lattice': 10.0, 'cost_composition': 1.0, 'cost_edge': 10.0, 'cost_property': 1.0,
                'beta': 0.01,
                'teacher_forcing_lattice': True,
                'teacher_forcing_max_epoch': 1000,
                'decoder': 'egnn'}

chggen = CHGGen(
    hparams_dict = model_hparams, lattice_scaler = lattice_scaler, 
)

device = torch.device('cpu')
checkpoint_path = "/home/xzdai/ceder_group/material_dircovery/chggen_old/test_models/perov/trainer_perov.ckpt"
chggen = chggen.load_from_checkpoint(checkpoint_path = checkpoint_path)
chggen.lattice_scaler = lattice_scaler
chggen.to(device = device)


/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:655: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  targets = torch.tensor([d[key] for d in data_list])
/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:619: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


CHGGen(
  (mse_composition): MSELoss()
  (wnd): wND()
  (encoder3d): CHGNet_encoder(
    (composition_model): AtomRef(
      (fc): Linear(in_features=94, out_features=1, bias=False)
    )
    (graph_converter): CrystalGraphConverter(algorithm='legacy', atom_graph_cutoff=5, bond_graph_cutoff=3)
    (atom_embedding): AtomEmbedding(
      (embedding): Embedding(94, 64)
    )
    (bond_basis_expansion): BondEncoder(
      (rbf_expansion_ag): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
      (rbf_expansion_bg): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
    )
    (bond_embedding): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_ag): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_bg): Linear(in_features=9, out_features=64, bias=False)
    (angle_basis_expansion): AngleEncoder(
      (fourier_expansion): Fourier()
    )
    (angle_embedding): Linear(in_features=9, out_features=64, bias=False)
    (atom_con

In [4]:
def sample_composition(composition_prob, num_atoms):
    """Sample composition such that it exactly satisfies composition_prob.
    
    Args:
        composition_prob (Tensor): The composition probability. The shape should be
            (num of structures, MAX_ATOMIC_NUM).
        num_atoms (Tensor): The number of atoms for each structure. The shape should
            be (num of structures).
    
    Returns:
        all_sampled_comp (Tensor): The sampled composition. The shape should be
            (num of atoms).
    """
    all_sampled_comp = []

    for comp_prob, num_atom in zip(list(composition_prob), list(num_atoms)):
        comp_num = torch.round(comp_prob * num_atom)
        atom_type = torch.nonzero(comp_num, as_tuple=True)[0]
        atom_num = comp_num[atom_type].long()

        sampled_comp = atom_type.repeat_interleave(atom_num, dim=0)

        # if the rounded composition gives less atoms, sample the rest
        if sampled_comp.size(0) < num_atom:
            left_atom_num = num_atom - sampled_comp.size(0)

            left_comp_prob = comp_prob - comp_num.float() / num_atom

            left_comp_prob[left_comp_prob < 0.] = 0.
            left_comp = torch.multinomial(
                left_comp_prob, num_samples=left_atom_num, replacement=True)
            # convert to atomic number
            left_comp = left_comp + 1
            sampled_comp = torch.cat([sampled_comp, left_comp], dim=0)

        sampled_comp = sampled_comp[torch.randperm(sampled_comp.size(0))]
        sampled_comp = sampled_comp[:num_atom]
        all_sampled_comp.append(sampled_comp)

    all_sampled_comp = torch.cat(all_sampled_comp, dim=0)
    assert all_sampled_comp.size(0) == num_atoms.sum()
    return all_sampled_comp


In [5]:
z = torch.randn(1, 64, requires_grad= True, device = device)
(num_atoms, 
pred_lengths_and_angles, 
pred_lengths, 
pred_angles, 
composition) = chggen.decode_stats(z)
num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles, composition[0]

/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:630: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


(tensor([5]),
 tensor([[ 3.5760e-01,  3.6039e-01,  3.8093e-01,  6.2322e-05,  1.3666e-04,
          -2.0495e-03]], grad_fn=<AddmmBackward0>),
 tensor([[4.2800, 4.2809, 4.2879]]),
 tensor([[90., 90., 90.]]),
 tensor([7.5745e-07, 1.7421e-06, 2.7495e-05, 9.6938e-05, 4.0884e-05, 1.2325e-06,
         5.3372e-02, 3.2603e-01, 6.0174e-02, 9.0300e-07, 5.2461e-03, 5.3048e-03,
         1.8906e-03, 5.8468e-04, 1.6214e-06, 7.1017e-02, 1.4454e-06, 8.6728e-07,
         9.4510e-04, 1.3209e-02, 1.2169e-02, 1.2346e-02, 9.8866e-03, 7.9793e-03,
         7.0120e-03, 2.9569e-03, 3.1571e-03, 1.3600e-03, 1.5992e-03, 4.0315e-03,
         8.4078e-03, 8.6948e-03, 1.1316e-02, 1.6212e-06, 2.7969e-06, 8.1696e-07,
         1.3233e-03, 9.9815e-03, 1.1004e-02, 3.2332e-02, 3.0125e-02, 1.7638e-02,
         1.6130e-06, 2.1908e-03, 5.9555e-03, 3.5916e-03, 4.6002e-03, 7.9880e-03,
         2.3508e-02, 2.4907e-02, 1.7055e-02, 1.6777e-02, 6.0441e-07, 8.2469e-07,
         1.8543e-03, 1.0124e-02, 1.9220e-02, 1.3072e-06, 1.1791e-

In [6]:
sample_composition(composition, num_atoms)

tensor([31, 51, 16,  7,  7])

In [7]:
from types import SimpleNamespace

ld_kwargs = SimpleNamespace(
    n_step_each = 30,
    step_lr = 1e-7,
    min_sigma = 0,
    save_traj = False,
    disable_bar = False,
    compute_force = True,
    beta_c = 0,         # property update rate
    beta_f = 0,         # atomic force update rate                          
)
ld_kwargs.n_step_each

30

In [8]:
results = chggen.langevin_dynamics_guidance(
    z = z, 
    prop_guidance = torch.tensor(-0.05, device= device), 
    ld_kwargs= ld_kwargs
)

  0%|          | 0/10 [00:00<?, ?it/s]

sigma step: 0
step: 0
YHgSO2
[Structure Summary
Lattice
    abc : 4.279970169067387 4.280911922454842 4.287853717803955
 angles : 90.000002504478 90.00000250447815 89.9999974955219
 volume : 78.56280758346291
      A : 4.279970169067383 0.0 -1.8708344384776865e-07
      B : 1.8712459848302387e-07 4.280911922454834 -1.8712459848302387e-07
      C : 0.0 0.0 4.287853717803955
    pbc : True True True
PeriodicSite: O (1.218, 0.2083, 0.9282) [0.2846, 0.04866, 0.2165]
PeriodicSite: Y (2.641, 0.4613, 1.709) [0.6171, 0.1078, 0.3986]
PeriodicSite: Hg (4.135, 1.013, 2.935) [0.9661, 0.2367, 0.6845]
PeriodicSite: O (3.789, 0.7344, 0.6272) [0.8852, 0.1715, 0.1463]
PeriodicSite: S (3.054, 1.309, 3.716) [0.7136, 0.3059, 0.8667]]
step: 1
YHgSO2
[Structure Summary
Lattice
    abc : 4.251869448034444 4.227897046597626 4.322826380773979
 angles : 90.08356213206123 90.83147914142818 90.42785814513164
 volume : 77.6986982190474
      A : 4.251425266265869 0.04162989556789398 -0.045210178941488266
      B :

 10%|█         | 1/10 [00:01<00:13,  1.47s/it]

sigma step: 1
step: 0
YHgSO2
[Structure Summary
Lattice
    abc : 4.672482141141594 4.2278930336117835 3.711084062330116
 angles : 97.15446904470355 91.96736930951609 86.58473006898576
 volume : 72.58475513618123
      A : 4.667459011077881 0.2165074497461319 0.006345344707369804
      B : 0.05698765441775322 4.216110706329346 -0.310229629278183
      C : -0.12379681318998337 -0.18925324082374573 3.7041871547698975
    pbc : True True True
PeriodicSite: O (3.058, 0.1215, 2.378) [0.672, 0.02316, 0.6428]
PeriodicSite: Y (0.2927, 0.1578, 1.224) [0.07098, 0.04878, 0.3343]
PeriodicSite: Hg (1.35, 0.8391, 1.076) [0.2949, 0.1976, 0.3064]
PeriodicSite: O (0.1145, 0.3745, 2.697) [0.04265, 0.1198, 0.7381]
PeriodicSite: S (2.908, 1.808, -0.08588) [0.6185, 0.3975, 0.009046]]
step: 1
YHgSO2
[Structure Summary
Lattice
    abc : 4.571712867422019 4.224179890085296 3.721531198597988
 angles : 98.00258474619451 91.79576487774197 86.2523851425408
 volume : 70.99918554168042
      A : 4.564917087554932 0

 20%|██        | 2/10 [00:02<00:11,  1.43s/it]

...

step: 27
YHgSO2
[Structure Summary
Lattice
    abc : 4.48978604798607 4.068064748728362 3.727545309668031
 angles : 99.4482526088763 95.08755227754392 89.45660780708197
 volume : 66.89358142713444
      A : 4.486976146697998 0.14803482592105865 -0.057528305798769
      B : -0.09983754903078079 4.049201965332031 -0.37834733724594116
      C : -0.27414989471435547 -0.2751149535179138 3.7072560787200928
    pbc : True True True
PeriodicSite: O (1.351, 3.185, 2.021) [0.3581, 0.8165, 0.6341]
PeriodicSite: Y (4.123, 0.6357, 3.448) [0.9819, 0.1866, 0.9644]
PeriodicSite: Hg (0.4837, 1.607, 0.7207) [0.1314, 0.4083, 0.2381]
PeriodicSite: O (3.798, 0.843, 2.416) [0.8935, 0.2223, 0.6881]
PeriodicSite: S (2.41, 1.443, 0.09843) [0.5491, 0.3409, 0.06986]]
step: 28
YHgSO2
[Structure Summary
Lattice
    abc : 4.488742011532235 4.068699418687182 3.728127540950235
 angles : 99.443473232073 95.0736909578102 89.49763089122271
 volume : 66.90102456613982
      A : 4.48599100112915 0.14631852507591248 -0.057

 90%|█████████ | 9/10 [00:13<00:01,  1.51s/it]

sigma step: 9
step: 0
YHgSO2
[Structure Summary
Lattice
    abc : 4.48755820003134 4.072399555767733 3.7377588108388378
 angles : 99.40304540403176 94.99294842270817 89.49620686275799
 volume : 67.13346492058346
      A : 4.484827041625977 0.14706742763519287 -0.053630001842975616
      B : -0.10158692300319672 4.053844451904297 -0.3747844696044922
      C : -0.2719820439815521 -0.27658402919769287 3.7175755500793457
    pbc : True True True
PeriodicSite: O (1.372, 3.171, 2.042) [0.363, 0.8124, 0.6365]
PeriodicSite: Y (4.126, 0.6593, 3.459) [0.9828, 0.1928, 0.9639]
PeriodicSite: Hg (0.4745, 1.628, 0.7386) [0.1299, 0.4134, 0.2422]
PeriodicSite: O (3.775, 0.833, 2.413) [0.8882, 0.2199, 0.6841]
PeriodicSite: S (2.402, 1.452, 0.08628) [0.5474, 0.3428, 0.06566]]
step: 1
YHgSO2
[Structure Summary
Lattice
    abc : 4.487386312519595 4.072923277276478 3.7372865180144728
 angles : 99.39693500411646 94.97389885259172 89.50975946036102
 volume : 67.13410833445742
      A : 4.484684944152832 0.146

100%|██████████| 10/10 [00:15<00:00,  1.51s/it]

step: 27
YHgSO2
[Structure Summary
Lattice
    abc : 4.485613283164865 4.069885043377816 3.73889458435988
 angles : 99.43339035412252 94.99854702667551 89.57951715172501
 volume : 67.07619629948427
      A : 4.4829607009887695 0.14420349895954132 -0.05472869798541069
      B : -0.10502978414297104 4.050990581512451 -0.37737026810646057
      C : -0.27167782187461853 -0.2762886583805084 3.718761682510376
    pbc : True True True
PeriodicSite: O (1.373, 3.151, 2.026) [0.3635, 0.808, 0.6321]
PeriodicSite: Y (4.132, 0.6553, 3.469) [0.9847, 0.1927, 0.9669]
PeriodicSite: Hg (0.47, 1.64, 0.7304) [0.1292, 0.4167, 0.2406]
PeriodicSite: O (3.8, 0.8303, 2.4) [0.894, 0.2196, 0.6809]
PeriodicSite: S (2.404, 1.453, 0.09158) [0.5483, 0.3437, 0.06757]]
step: 28
YHgSO2
[Structure Summary
Lattice
    abc : 4.485592367017267 4.069278298518433 3.738781826011392
 angles : 99.4321831945401 94.98889831143582 89.59297043639603
 volume : 67.0649822696623
      A : 4.482962608337402 0.14334578812122345 -0.05511

In [9]:
lattices = results['lattices']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

print(lattices.shape)
print(num_atoms)
print(frac_coords.shape)
print(atom_types.shape)

torch.Size([1, 3, 3])
tensor([5])
torch.Size([5, 3])
torch.Size([5])


In [10]:
from pymatgen.core import Structure, Lattice, Element

In [11]:
s = Structure(
    lattice = lattices[0].detach().numpy(), 
    species = atom_types.detach().numpy(), 
    coords = frac_coords.detach().numpy(),
    to_unit_cell = False,
    coords_are_cartesian = False,
)

In [12]:
s.to(filename = 'test.cif')

"# generated using pymatgen\ndata_YHgSO2\n_symmetry_space_group_name_H-M   'P 1'\n_cell_length_a   4.48658883\n_cell_length_b   4.06868012\n_cell_length_c   3.73890155\n_cell_angle_alpha   99.42164545\n_cell_angle_beta   95.00605153\n_cell_angle_gamma   89.59145860\n_symmetry_Int_Tables_number   1\n_chemical_formula_structural   YHgSO2\n_chemical_formula_sum   'Y1 Hg1 S1 O2'\n_cell_volume   67.07245646\n_cell_formula_units_Z   1\nloop_\n _symmetry_equiv_pos_site_id\n _symmetry_equiv_pos_as_xyz\n  1  'x, y, z'\nloop_\n _atom_site_type_symbol\n _atom_site_label\n _atom_site_symmetry_multiplicity\n _atom_site_fract_x\n _atom_site_fract_y\n _atom_site_fract_z\n _atom_site_occupancy\n  O  O0  1  0.36320969  0.80751961  0.63212562  1\n  Y  Y1  1  0.98413211  0.19265604  0.96611631  1\n  Hg  Hg2  1  0.12964812  0.41872951  0.24090974  1\n  O  O3  1  0.89456922  0.22025092  0.68278366  1\n  S  S4  1  0.54871869  0.34300634  0.06805822  1\n"